# Cuaderno para hacer fine-tuning


## Configuración del cuaderno


In [ ]:
#Instalación de paquetes
!pip install tf_keras tensorflow numpy matplotlib -q
!pip install coral-ordinal

import os

# Forzar uso de Keras 2 para evitar problemas de compatibilidad con STM32Cube.AI
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import tf_keras as keras
from tf_keras import layers, Model
import coral_ordinal as coral


# Verificar que estamos en Keras 2
assert int(keras.__version__.split('.')[0]) == 2, "No se está usando la versión de Keras2"
print("Usando Keras2")

from google.colab import drive
drive.mount('/content/drive')

import sys

SRC_PATH = "/content/drive/MyDrive/TFG/src"

if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

In [ ]:
# Macros

TRAIN_PATH = '/content/drive/MyDrive/TFG/dataset/train'
VAL_PATH = '/content/drive/MyDrive/TFG/dataset/val'

CHECKPOINT_PATH = '/content/drive/MyDrive/TFG/checkpoints'
MODEL_PATH = '/content/drive/MyDrive/TFG/modelos'

os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

IMAGE_SIZE = (480, 270)
INPUT_SHAPE = (224, 224, 3)

NUM_BATCHES = 32
NUM_EPOCH = 50

LABELS = {'0':'fluido', '1' : 'moderado', '2' : 'denso', '3' : 'saturado'}
NUM_CLASES = len(LABELS)


## Funciones auxiliares

In [ ]:
# Preprocesado de imágenes
from preprocesado import preprocesado, ImageCropY
preprocesado = preprocesado(IMAGE_SIZE)

## Dataset

In [ ]:
# Carga del dataset
train_ds = keras.utils.image_dataset_from_directory (
    directory = TRAIN_PATH,
    labels = 'inferred',
    label_mode = 'int',
    batch_size = NUM_BATCHES,
    image_size = IMAGE_SIZE
)

val_ds = keras.utils.image_dataset_from_directory (
    directory = VAL_PATH,
    labels = 'inferred',
    label_mode = 'int',
    batch_size = NUM_BATCHES,
    image_size = IMAGE_SIZE
)

# Preprocesado del dataset
train_ds = train_ds.map(
    lambda x, y: (preprocesado(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

val_ds = val_ds.map(
    lambda x, y: (preprocesado(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

## Fine-tuning



In [ ]:
# Carga del modelo
CORAL_CUSTOM_OBJECTS = {
    'CoralOrdinal'           : coral.CoralOrdinal,
    'OrdinalCrossEntropy'    : coral.OrdinalCrossEntropy,
    'MeanAbsoluteErrorLabels': coral.MeanAbsoluteErrorLabels,
    'ImageCropY'             : ImageCropY
}

model = keras.models.load_model(
    f'{MODEL_PATH}/modelo_final.keras',
    custom_objects=CORAL_CUSTOM_OBJECTS
)
print("Modelo cargado correctamente")

# Descongelar últimas capas
base_model = model.get_layer('mobilenet_0.50_224')
base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

print(f"Capas totales en base_model: {len(base_model.layers)}")
print(f"Trainable weights antes de recompilar: {len(model.trainable_weights)}")

trainable_params = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f"Trainable params: {trainable_params:,}")

# Compilación del modelo
model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=coral.OrdinalCrossEntropy(num_classes=NUM_CLASES),
    metrics=[coral.MeanAbsoluteErrorLabels(name='mae_labels')]
)

model.summary()

In [ ]:
# Definición de los callbacks
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath = f'{CHECKPOINT_PATH}/best_model_ft.keras',
        monitor='val_mae_labels',
        save_best_only=True,
        mode = 'min',
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_mae_labels',
        patience=12,
        restore_best_weights=True,
        mode = 'min',
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_mae_labels',
        factor=0.5,
        patience=6,
        min_lr = 1e-7,
        mode = 'min',
        verbose=1
    )
]

# Fine-tuning
history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks
)

# Guardado del modelo
model.save(f'{MODEL_PATH}/modelo_ft_final.keras')
print("Modelo guardado en: ", MODEL_PATH)